# @traceable 的 run_type 与消息格式要求

`@traceable` 装饰器的 `run_type` 参数控制 LangSmith 如何解析、渲染该 Run 的输入/输出。

下表汇总了所有合法的 `run_type` 取值，以及各自对 **输入格式**、**输出格式** 和 **特殊字段** 的要求。

## Setup

In [1]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

## run_type 完整对照表

| run_type | 含义 | 输入格式要求 | 输出格式要求 | 特殊字段 / 说明 |
|---|---|---|---|---|
| `"llm"` | 调用语言模型 | 函数参数必须命名为 `messages`，值为包含 `role` + `content` 的 dict 列表（OpenAI 兼容格式） | 以下四种之一：① `{"choices": [{"message": {"role":..., "content":...}}]}`；② `{"message": {"role":..., "content":...}}`；③ `{"role":..., "content":...}`；④ `[role, content]` 元组 | `metadata` 中可传 `ls_provider`（如 `"openai"`）和 `ls_model_name`（如 `"gpt-4o-mini"`）以启用费用估算；输出中可附 `usage_metadata`（含 `input_tokens`/`output_tokens`/`total_tokens`）；流式输出需配合 `reduce_fn` 合并 chunk |
| `"retriever"` | 从向量库/知识库检索文档 | 无特殊要求，通常传入查询字符串 | 必须返回 `list[dict]`，每个 dict 包含 `page_content`（str）、`type`（固定为 `"Document"`）、`metadata`（dict） | 若格式不符，数据仍会记录但 LangSmith UI 不会启用文档专属渲染 |
| `"tool"` | 执行工具/函数调用 | 无特殊要求，传入函数原始参数即可 | 无特殊要求，返回任意可序列化值 | 纯语义标注，UI 显示为工具图标；通常在 Agent 场景中使用 |
| `"chain"` | 编排多个子 Run 的流程（**默认值**） | 无特殊要求 | 无特殊要求 | 默认 run_type；所有未显式指定类型的 `@traceable` 函数均为此类型 |
| `"prompt"` | 构造/渲染提示词 | 无特殊要求，通常传入模板变量 | 无特殊要求，通常返回格式化后的提示字符串 | 纯语义标注，用于标识提示词构建步骤 |
| `"parser"` | 解析 LLM 原始输出 | 无特殊要求，通常传入待解析的字符串 | 无特殊要求，通常返回结构化数据（dict/list 等） | 纯语义标注，用于标识输出解析步骤 |
| `"embedding"` | 生成向量嵌入 | 无特殊要求，通常传入待嵌入的文本或文本列表 | 无特殊要求，通常返回浮点数列表或二维列表 | 纯语义标注，用于标识嵌入计算步骤 |

## 代码示例

下面用具体代码演示每种 `run_type` 的标准用法，重点关注**格式敏感**的类型。

### 1. `run_type="llm"` —— 格式要求最严格

- 参数名必须是 `messages`
- 支持四种输出格式（见下方注释）
- `metadata` 中的 `ls_provider` / `ls_model_name` 用于 LangSmith 费用估算

In [2]:
from langsmith import traceable

# 输入：OpenAI 兼容的消息列表，参数名必须是 messages
llm_inputs = [
    {"role": "system", "content": "你是一个有用的助手。"},
    {"role": "user",   "content": "今天天气如何？"},
]

# 输出格式①：choices 列表（最常用，与 OpenAI SDK 返回格式一致）
llm_output_choices = {
    "choices": [
        {"message": {"role": "assistant", "content": "今天阳光明媚！"}}
    ]
}

# 输出格式②：单条 message
# llm_output_message = {"message": {"role": "assistant", "content": "今天阳光明媚！"}}

# 输出格式③：直接含 role + content
# llm_output_direct = {"role": "assistant", "content": "今天阳光明媚！"}

# 输出格式④：(role, content) 元组
# llm_output_tuple = ("assistant", "今天阳光明媚！")

@traceable(
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-4o-mini",
    },
)
def mock_chat_model(messages: list) -> dict:
    """模拟 LLM 调用，演示 run_type='llm' 的消息格式。"""
    return llm_output_choices

result = mock_chat_model(llm_inputs)
print("LLM 输出：", result)

LLM 输出： {'choices': [{'message': {'role': 'assistant', 'content': '今天阳光明媚！'}}]}


### 1b. `run_type="llm"` 流式输出 —— 使用 `reduce_fn`

流式场景下，每次 `yield` 一个 chunk（格式与非流式输出相同），
并通过 `reduce_fn` 将所有 chunk 合并为最终输出记录到 LangSmith。

In [3]:
from langsmith import traceable

def _reduce_chunks(chunks: list) -> dict:
    """将多个流式 chunk 合并为单条完整输出。"""
    full_text = "".join(
        chunk["choices"][0]["message"]["content"] for chunk in chunks
    )
    return {"choices": [{"message": {"role": "assistant", "content": full_text}}]}

@traceable(
    run_type="llm",
    metadata={"ls_provider": "mock", "ls_model_name": "mock-stream"},
    reduce_fn=_reduce_chunks,  # 流式场景必须提供，否则 LangSmith 只记录原始 chunk 列表
)
def mock_streaming_chat_model(messages: list):
    """逐 token 流式返回，每个 chunk 格式与非流式输出格式①相同。"""
    tokens = ["你好", "，", "今天", "阳光明媚！"]
    for token in tokens:
        yield {"choices": [{"message": {"role": "assistant", "content": token}}]}

chunks = list(mock_streaming_chat_model(llm_inputs))
print("流式 chunk 数量：", len(chunks))
print("合并后文本：", "".join(c["choices"][0]["message"]["content"] for c in chunks))

流式 chunk 数量： 4
合并后文本： 你好，今天阳光明媚！


### 2. `run_type="retriever"` —— 输出格式必须含固定字段

每个文档 dict 必须包含：
- `page_content`：文档正文（字符串）
- `type`：固定为 `"Document"`
- `metadata`：附加信息 dict（可为空 dict）

In [4]:
from langsmith import traceable

@traceable(run_type="retriever")
def mock_retriever(query: str) -> list:
    """模拟向量库检索，演示 run_type='retriever' 的输出格式。"""
    # 每个文档必须包含 page_content、type="Document"、metadata 三个字段
    return [
        {
            "page_content": "LangSmith 是 LangChain 推出的可观测性平台。",
            "type": "Document",
            "metadata": {"source": "docs.smith.langchain.com", "chunk_id": 0},
        },
        {
            "page_content": "通过 @traceable 可以将任意函数的输入/输出记录到 LangSmith。",
            "type": "Document",
            "metadata": {"source": "docs.smith.langchain.com", "chunk_id": 1},
        },
    ]

docs = mock_retriever("什么是 LangSmith？")
print(f"检索到 {len(docs)} 条文档")
for doc in docs:
    print(" -", doc["page_content"][:30], "...")

检索到 2 条文档
 - LangSmith 是 LangChain 推出的可观测性平 ...
 - 通过 @traceable 可以将任意函数的输入/输出记录到 ...


### 3. `run_type="tool"` —— 纯语义标注，无格式约束

In [5]:
from langsmith import traceable

@traceable(run_type="tool")
def get_weather(location: str, unit: str = "Celsius") -> dict:
    """模拟天气查询工具，演示 run_type='tool' 的用法。"""
    # 输入/输出格式完全自由，LangSmith 只负责语义标注
    return {"location": location, "temperature": 22, "unit": unit, "condition": "晴"}

weather = get_weather("北京", "Celsius")
print("天气信息：", weather)

天气信息： {'location': '北京', 'temperature': 22, 'unit': 'Celsius', 'condition': '晴'}


### 4. `run_type="chain"` —— 默认值，编排多个子 Run

In [6]:
from langsmith import traceable

@traceable(run_type="chain")  # 等同于直接用 @traceable（默认即为 chain）
def rag_pipeline(question: str) -> str:
    """将检索、生成等子步骤串联为一条完整 trace。"""
    docs = mock_retriever(question)
    context = "\n".join(d["page_content"] for d in docs)
    # 实际场景中此处调用 LLM，这里直接返回 context 演示结构
    return f"[基于检索结果的回答]\n{context[:60]}..."

answer = rag_pipeline("什么是 LangSmith？")
print(answer)

[基于检索结果的回答]
LangSmith 是 LangChain 推出的可观测性平台。
通过 @traceable 可以将任意函数的输入/输出...


### 5. `run_type="prompt"` —— 标注提示词构建步骤

In [7]:
from langsmith import traceable

@traceable(run_type="prompt")
def build_prompt(context: str, question: str) -> list:
    """将检索到的上下文和用户问题组装为消息列表。"""
    return [
        {"role": "system", "content": "你是一个基于给定上下文回答问题的助手。"},
        {"role": "user",   "content": f"上下文：{context}\n\n问题：{question}"},
    ]

prompt = build_prompt("LangSmith 是可观测性平台。", "LangSmith 有什么功能？")
print("构建的提示词：")
for msg in prompt:
    print(f"  [{msg['role']}] {msg['content'][:40]}...")

构建的提示词：
  [system] 你是一个基于给定上下文回答问题的助手。...
  [user] 上下文：LangSmith 是可观测性平台。

问题：LangSmith 有什么...


### 6. `run_type="parser"` —— 标注输出解析步骤

In [8]:
import json
from langsmith import traceable

@traceable(run_type="parser")
def parse_json_output(raw_output: str) -> dict:
    """将 LLM 返回的 JSON 字符串解析为结构化 dict。"""
    # 输入通常是 LLM 的原始文本输出，输出是解析后的结构化数据
    return json.loads(raw_output)

raw = '{"name": "张三", "age": 28, "city": "上海"}'
parsed = parse_json_output(raw)
print("解析结果：", parsed)

解析结果： {'name': '张三', 'age': 28, 'city': '上海'}


### 7. `run_type="embedding"` —— 标注向量化步骤

In [9]:
import random
from langsmith import traceable

@traceable(run_type="embedding")
def mock_embed(texts: list[str]) -> list[list[float]]:
    """模拟文本向量化，演示 run_type='embedding' 的用法。"""
    # 实际场景中调用 OpenAI Embeddings / sentence-transformers 等
    return [[random.random() for _ in range(8)] for _ in texts]

vectors = mock_embed(["LangSmith", "向量检索"])
print(f"生成 {len(vectors)} 个向量，维度 {len(vectors[0])}")

生成 2 个向量，维度 8


## 通用能力：`process_inputs` / `process_outputs`

所有 `run_type` 均支持在记录前对输入/输出做变换（脱敏、截断、格式转换等），
不影响函数的实际运行逻辑。

In [10]:
from langsmith import traceable

@traceable(
    run_type="chain",
    # 记录前截断过长的输入，避免 trace 数据过大
    process_inputs=lambda inputs: {
        k: (v[:100] + "..." if isinstance(v, str) and len(v) > 100 else v)
        for k, v in inputs.items()
    },
    # 记录前只保留关键字段
    process_outputs=lambda output: {"summary": str(output)[:200]},
)
def process_long_text(text: str) -> str:
    """演示 process_inputs / process_outputs 对 trace 数据的预处理。"""
    return f"已处理：{text[:20]}..."

result = process_long_text("这是一段很长的文本，" * 20)
print(result)

已处理：这是一段很长的文本，这是一段很长的文本，...


## 小结

| 格式严格程度 | run_type | 关键约束 |
|---|---|---|
| **严格** | `llm` | 参数名必须为 `messages`；输出必须是四种格式之一；流式需 `reduce_fn` |
| **半严格** | `retriever` | 输出必须含 `page_content`、`type="Document"`、`metadata` 字段 |
| **无约束** | `tool` / `chain` / `prompt` / `parser` / `embedding` | 纯语义标注，输入/输出格式完全自由 |

> **最佳实践**：即使是「无约束」的 run_type，也建议保持输入/输出可 JSON 序列化，
> 否则 LangSmith SDK 在序列化时会静默跳过无法处理的字段。